# 06 - Cruce Tópicos × Escala Likert

**Grupo 15 | UMSS FCyT | IA I/2026**

La calificación Likert actúa como **proxy de polaridad**:
cruzarla con los tópicos descubiertos permite identificar
qué dimensiones generan insatisfacción y cuáles satisfacción,
sin necesidad de un modelo de sentimientos externo.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('Librerías cargadas ✅')

## 1. Cargar dataset con tópicos asignados

In [ ]:
import os

RUTA = '../data/processed/encuesta_con_topicos.csv'
if os.path.exists(RUTA):
    df = pd.read_csv(RUTA)
else:
    # Si no existe el archivo procesado, usar el original
    df = pd.read_csv('../data/raw/encuesta_sintetica.csv')
    print('⚠️  Ejecuta primero el notebook 04 para obtener tópicos asignados.')

df = df[df['topico_dominante'] >= 0].copy()
print(f'Registros con tópico asignado: {len(df)}')
df.head()

## 2. Likert promedio por tópico dominante

In [ ]:
avg = df.groupby('topico_dominante')['calificacion_likert'].agg(['mean', 'count', 'std'])
avg.columns = ['Media', 'N', 'Std']
print(avg.round(2))

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#d32f2f' if v < 3 else '#ffd600' if v < 4 else '#388e3c' for v in avg['Media']]
bars = ax.bar([f'Tópico {i}' for i in avg.index], avg['Media'], color=colors)
ax.axhline(3, color='gray', linestyle='--', alpha=0.7, label='Neutral (3)')
ax.axhline(df['calificacion_likert'].mean(), color='blue',
           linestyle=':', alpha=0.7, label=f'Media global ({df["calificacion_likert"].mean():.2f})')
ax.set_ylim(0, 5.5)
ax.set_ylabel('Promedio Likert')
ax.set_title('Satisfacción media por tópico dominante', fontweight='bold')
ax.legend()
for bar, n in zip(bars, avg['N']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f'n={n}',
            ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('../data/processed/fig_likert_por_topico.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Distribución Likert por tópico (stacked bar)

In [ ]:
pivot = df.groupby(['topico_dominante', 'calificacion_likert']).size().unstack(fill_value=0)

likert_labels = {1: 'Muy malo', 2: 'Malo', 3: 'Regular', 4: 'Bueno', 5: 'Muy bueno'}
likert_colors = ['#d32f2f', '#f57c00', '#ffd600', '#388e3c', '#1565c0']

# Normalizar a porcentaje
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(10, 5))
bottom = pd.Series([0] * len(pivot_pct), index=pivot_pct.index)
for col in pivot_pct.columns:
    ax.bar(pivot_pct.index, pivot_pct[col], bottom=bottom,
           label=likert_labels.get(col, col),
           color=likert_colors[col - 1] if col <= 5 else 'gray')
    bottom += pivot_pct[col]

ax.set_xticks(pivot_pct.index)
ax.set_xticklabels([f'Tópico {i}' for i in pivot_pct.index])
ax.set_ylabel('% de respuestas')
ax.set_title('Distribución de calificaciones Likert por tópico', fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), title='Likert')
plt.tight_layout()
plt.savefig('../data/processed/fig_likert_stacked.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Análisis por docente

In [ ]:
if 'docente' in df.columns:
    avg_doc = df.groupby('docente')['calificacion_likert'].mean().sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(8, 4))
    col_doc = ['#388e3c' if v >= 4 else '#ffd600' if v >= 3 else '#d32f2f' for v in avg_doc.values]
    ax.bar(avg_doc.index, avg_doc.values, color=col_doc)
    ax.axhline(3, color='gray', linestyle='--', alpha=0.6)
    ax.set_ylim(0, 5.5)
    ax.set_ylabel('Promedio Likert')
    ax.set_title('Calificación promedio por docente', fontweight='bold')
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.savefig('../data/processed/fig_likert_docentes.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Tabla detallada
    tabla_doc = df.groupby('docente')['calificacion_likert'].agg(['mean', 'count', 'std']).round(2)
    tabla_doc.columns = ['Promedio', 'N comentarios', 'Desv. estándar']
    print(tabla_doc.sort_values('Promedio', ascending=False))

## 5. Análisis por dimensión × Likert

In [ ]:
if 'dimension' in df.columns:
    pivot_dim = df.pivot_table(
        values='calificacion_likert',
        index='docente',
        columns='dimension',
        aggfunc='mean'
    )

    fig, ax = plt.subplots(figsize=(10, 4))
    sns.heatmap(pivot_dim, annot=True, fmt='.1f', cmap='RdYlGn',
                vmin=1, vmax=5, linewidths=0.5, ax=ax)
    ax.set_title('Likert promedio: Docente × Dimensión', fontweight='bold')
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.savefig('../data/processed/fig_heatmap_docente_dim.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. Conclusiones

In [ ]:
print('=== RESUMEN DE HALLAZGOS ===')
print(f'Calificación global promedio: {df["calificacion_likert"].mean():.2f} / 5')
print()

avg_top = df.groupby('topico_dominante')['calificacion_likert'].mean()
mejor = avg_top.idxmax()
peor  = avg_top.idxmin()
print(f'Tópico con mayor satisfacción : Tópico {mejor} (μ={avg_top[mejor]:.2f})')
print(f'Tópico con menor satisfacción : Tópico {peor}  (μ={avg_top[peor]:.2f})')

if 'docente' in df.columns:
    avg_doc = df.groupby('docente')['calificacion_likert'].mean()
    print(f'Docente mejor valorado  : {avg_doc.idxmax()} (μ={avg_doc.max():.2f})')
    print(f'Docente menor valorado  : {avg_doc.idxmin()} (μ={avg_doc.min():.2f})')